In [1]:
import os 
import sys
from pathlib import Path

import numpy as np 
import torch 
from tqdm import tqdm 
from einops import rearrange 

from eb_jepa.vis_utils import show_images,analyze_distances,plot_losses

In [2]:
PKG_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "cfgs" / "train.yaml").exists()
)
sys.path.insert(0, str(PKG_ROOT))

from planning import GCAgent
from builders import build_train_cfg, build_eval_cfg, build_plan_cfg, build_model

### Cfgs & JEPA

In [3]:
cfg, loader, val_loader, data_config = build_train_cfg()
eval_cfg, env_config, env_creator = build_eval_cfg()
env = env_creator()

jepa, xy_prober = build_model(
    cfg, 
    data_config=data_config, 
    normalizer=loader.dataset.normalizer
    )

plan_cfg = build_plan_cfg(logging_cfg=cfg.logging)
plan_cfg.planner.planner_name

[INFO    ][2026-09-05 08:43:47][eb_jepa.training_utils][load_config              ] Loaded config from /Users/hawardizayee/Desktop/AMI/eb_jepa/examples/my_ac_video_jepa/cfgs/train.yaml
[INFO    ][2026-09-05 08:43:47][eb_jepa.training_utils][load_config              ] Loaded config from /Users/hawardizayee/Desktop/AMI/eb_jepa/examples/my_ac_video_jepa/cfgs/eval.yaml
[INFO    ][2026-09-05 08:43:47][eb_jepa.training_utils][load_config              ] Loaded config from /Users/hawardizayee/Desktop/AMI/eb_jepa/examples/my_ac_video_jepa/cfgs/planning_mppi.yaml


/Users/hawardizayee/Desktop/AMI/eb_jepa/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 10 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


'mppi'

# `Main_eval`

In [4]:
env = env_creator()

agent = GCAgent(
    jepa,
    action_dim=2,
    plan_cfg=plan_cfg,
    normalizer=env.normalizer,
    loc_prober=xy_prober,
    env=env
)

successes = []
distances = []
episode_times = []
episode_observations = []
episode_infos = []

for ep in range(1):
    print(f'============ {ep} ================')
    ep_folder = Path("plan_eval") / f"ep_{ep}"
    os.makedirs(ep_folder, exist_ok=True)
    obs, info = env.reset() 
    obs, reward, done, truncated, info = env.step(
        np.zeros(env.action_space.shape[0]) # arrya([0, 0])
    ) # step with zero action to get the first observation
    goal_img = info["target_obs"]   

    combined = torch.stack([obs, goal_img], dim = 0) # (2,C=2,H=65,W=65)
    # shows init and goal images 
    # show_images(
    #     combined,
    #     nrow=2,
    #     titles=["Init", "Goal"],
    #     save_path=f"ep_{ep}_state.pdf",
    #     close_fig=True,
    #     first_channel_only=False,
    #     clamp=False
    # )

    agent.set_goal(
        goal_state=goal_img.detach().clone().to(dtype=torch.float32),
        goal_position=info["target_position"]
    )

    done = False # this is not used 
    steps_left = env.n_allowed_steps    # starts with 200 
    steps_left = 5  # overwritting 


    t0 = True 
    observations = [obs]
    infos = [info]

    prev_losses = []
    prev_elite_losses_mean = []
    prev_elite_losses_std = []

    while steps_left > 0:
        print(f'step : {5 - steps_left}')
        # while (not done and steps_left > 0 ):
        # plan_vis_path = (
        #     f"{ep_plan_vis_dir}/step{env.n_allowed_steps - steps_left}"
        #     if agent.decode_each_iteration  # False
        #     else None 
        # )
        plan_vis_path = None 
        # first loop iter: obs is from reset(), then it is from step()
        obs_tensor = (
            env.normalizer.normalize_state(
                obs.detach().clone().to(dtype=torch.float32, device=agent.device)
            )
            .unsqueeze(0)
            .unsqueeze(2)
        )
        
        with torch.no_grad():
            action = (
                agent.act(
                    obs_tensor,
                    steps_left=steps_left,
                    t0=t0,
                    plan_vis_path=plan_vis_path
                )
                .cpu()
                .numpy()
            )   # T, A

        if agent._prev_losses is not None: 
            prev_losses.append(agent._prev_losses)
            prev_elite_losses_mean.append(agent._prev_elite_losses_mean)
            prev_elite_losses_std.append(agent._prev_elite_losses_std)
        for a in action:
            obs, reward, done, truncated, info = env.step(a) 
            t0 = False 
            observations.append(obs)
            infos.append(info)
            steps_left -= 1 
            eval_results = env.eval_state(info["target_position"], info["dot_position"])
            success = eval_results["success"]
            state_dist = eval_results["state_dist"]
            print(f'success : {success} | state_dist : {state_dist}')

           
    episode_observations.append(torch.stack(observations))  
    episode_infos.append(infos)
    successes.append(success)
    distances.append(state_dist)

    # Analyze data 
    analyze_distances(
        episode_observations[-1],
        episode_infos[-1],
        str(ep_folder / "agent"),
        goal_position=agent.goal_position,
        goal_state=agent.goal_state,
        normalizer=agent.normalizer,
        model=agent.model,
        objective=agent.objective,
        device=agent.device
    )

    plot_losses(
        prev_losses,
        prev_elite_losses_mean,
        prev_elite_losses_std,
        work_dir=ep_folder,
        num_act_stepped=agent.num_act_stepped
    )

============ 0 ================
step : 0
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 6, 1, 1])
predicted_states : torch.Size([200, 512, 

In [5]:
print(len(episode_observations))
print(episode_observations[0].shape)

1
torch.Size([6, 2, 65, 65])


In [6]:
task_data = {
    "success_rate" : np.mean(successes),
    "mean_state_dist": np.mean(distances),
}

print(task_data["success_rate"])
print(task_data["mean_state_dist"])

0.0
28.004905533213993


# Running a single episode 

In [7]:
env = env_creator()

agent = GCAgent(
    jepa,
    action_dim=2,
    plan_cfg=plan_cfg,
    normalizer=env.normalizer,
    loc_prober=xy_prober,
    env=env
)

In [8]:
# start of an episode 
obs,info = env.reset()      
obs,reward,done, truncated, info = env.step(np.zeros(env.action_space.shape[0]))
goal_img = info["target_obs"] 
agent.set_goal(
    goal_state= goal_img.detach().clone().to(dtype=torch.float32),
    goal_position=info["target_position"]
)

steps_left = 3 
t0 = True 
observations = [obs]
infos = [info]

prev_losses = []
prev_elite_losses_mean = []
prev_elite_losses_std = []


while steps_left > 0:
    obs_tensor = (
        env.normalizer.normalize_state(
            obs.detach().clone().to(dtype=torch.float32, device=agent.device)
        )
        .unsqueeze(0)
        .unsqueeze(2)
    )       # (B=1, C=2, T=1, H=65, W=65)

    with torch.no_grad():
        action = (
            agent.act(
                obs_tensor,
                steps_left=steps_left,
                t0=t0
            )
        )       # (T=1 , A=2) # T is [:self.num_act_stepped=1] 

    # losses are set by act method to the agent attr
    prev_losses.append(agent._prev_losses)
    prev_elite_losses_mean.append(agent._prev_elite_losses_mean)
    prev_elite_losses_std.append(agent._prev_elite_losses_std)


    for a in action:        # there is only action
        obs, _, _, _, info = env.step(a)
        t0 = False 
        observations.append(obs)
        infos.append(info)
        steps_left -= 1 
        eval_results = env.eval_state(info["target_position"], info["dot_position"]) # taking l2
        success = eval_results["success"]        # overwritten (only the last one survives)
        state_dist = eval_results["state_dist"]  # same as above


    print(f'success : {success}     |   state_dis : {state_dist}')
    


    break

predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([200, 512, 4, 1, 1])
predicted_states : torch.Size([

/Users/hawardizayee/Desktop/AMI/eb_jepa/eb_jepa/datasets/two_rooms/env.py:139: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  action = torch.tensor(action, device=self.device)
